## **Problem Statement**

Organizations are increasingly recognizing the pivotal role customer feedback plays in shaping the trajectory of their products and services. The ability to swiftly and effectively respond to customer input not only fosters enhanced customer experiences but also serves as a catalyst for growth, prolonged customer engagement, and the nurturing of lifetime value relationships.

While organizations may be inundated with a wealth of customer-generated feedback and support tickets, the product management role entails much more than just processing the inputs.

To make efforts in managing customer experience and expectations truly impactful, it is necessary to adopt a structured approach – a method that allows to discern the most pressing issues, set priorities, and allocate resources judiciously. One of the most effective strategies is the power of Support Ticket Categorization.


### Objective

This exercise develops an advanced support ticket categorization system that accurately classifies incoming tickets, assigns relevant tags based on their content, implements mechanisms and generate the first response based on the sentiment for prioritizing tickets for prompt resolution.


## **Installing and Importing Libraries and Dependencies**

In [ ]:
# Installation for GPU llama-cpp-python

!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.45 --force-reinstall --no-cache-dir -q

In [1]:
# Installation for CPU llama-cpp-python

!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.45 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 147.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 197.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 310.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 235.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 195.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.5.3 which is incompatible.


**Note** : There may be an error related to a dependency issue thrown by the pip package. This can be ignored as it will not impact the execution of the code.

In [2]:
# For downloading the models from Hugging Face Hub
!pip install huggingface_hub==0.20.3 pandas==1.5.3 -q

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [3]:
# Function to download the model from the Hugging Face model hub
from huggingface_hub import hf_hub_download

# Importing the Llama class from the llama_cpp module
from llama_cpp import Llama

# Importing the json module
import json

# for loading and manipulating data
import pandas as pd

# for time computations
import time

import warnings
warnings.filterwarnings("ignore")

## **Loading the Data**

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rebeccaoshea/support-ticket-text-data-mid-term-nlp-project")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'support-ticket-text-data-mid-term-nlp-project' dataset.
Path to dataset files: /kaggle/input/support-ticket-text-data-mid-term-nlp-project


In [5]:
# Install if needed
!pip install -q kagglehub

import pandas as pd
from pathlib import Path

# Download the dataset
dataset_directory = Path(
    kagglehub.dataset_download(
        "rebeccaoshea/support-ticket-text-data-mid-term-nlp-project"
    )
)

print("Dataset directory:", dataset_directory)

# Find every CSV in the downloaded directory
csv_files = list(dataset_directory.rglob("*.csv"))

print("\nCSV files found:")
for file in csv_files:
    print(file)

Using Colab cache for faster access to the 'support-ticket-text-data-mid-term-nlp-project' dataset.
Dataset directory: /kaggle/input/support-ticket-text-data-mid-term-nlp-project

CSV files found:
/kaggle/input/support-ticket-text-data-mid-term-nlp-project/Support_ticket_text_data_mid_term.csv


In [6]:
csv_path = dataset_directory / "Support_ticket_text_data_mid_term.csv"

# Fallback in case the file is inside a subdirectory
if not csv_path.exists():
    matches = [
        file for file in csv_files
        if file.name.lower() == "Support_ticket_text_data_mid_term.csv"
    ]

    if not matches:
        raise FileNotFoundError(
            f"Support_ticket_text_data_mid_term.csv was not found. Available files: {csv_files}"
        )

    csv_path = matches[0]

print("Loading:", csv_path)

ticket_data = pd.read_csv(csv_path)

display(ticket_data.head())

Loading: /kaggle/input/support-ticket-text-data-mid-term-nlp-project/Support_ticket_text_data_mid_term.csv


,support_tick_id,support_ticket_text
0,ST2023-006,My internet connection has significantly slowe...
1,ST2023-007,Urgent help required! My laptop refuses to sta...
2,ST2023-008,I've accidentally deleted essential work docum...
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...
4,ST2023-010,"My smartphone battery is draining rapidly, eve..."


## **Data Overview**

In [10]:
# The first 5 rows of the data
ticket_data.head(5)

,support_tick_id,support_ticket_text
0,ST2023-006,My internet connection has significantly slowe...
1,ST2023-007,Urgent help required! My laptop refuses to sta...
2,ST2023-008,I've accidentally deleted essential work docum...
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...
4,ST2023-010,"My smartphone battery is draining rapidly, eve..."


In [11]:
# The shape of the data
ticket_data.shape

(21, 2)

In [12]:
# Check for missing values in the data
ticket_data.isnull().sum()

,0
support_tick_id,0
support_ticket_text,0


## **Model Building**

### Loading the model

In [13]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [14]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename

)

mistral-7b-instruct-v0.2.Q6_K.gguf: reconstructing file:   0%|          |  0.00B / 5.94GB            

mistral-7b-instruct-v0.2.Q6_K.gguf: downloading bytes:           |  0.00B            

In [15]:
llm = Llama(
     model_path=model_path,
     n_ctx=1024,
 )

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loade

In [16]:
#this is in case CPU is working instead of GPU

llm = Llama(
     model_path=model_path,
    n_ctx=1024, # Context window
    n_cores=-2 # Number of CPU cores to use
)

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loade

## **Ticket Categorization and Returning Structured Output**

In [17]:
# creating a copy of the data
data_1 = ticket_data.copy()

In [18]:
#Defining the response funciton for Task 1.
def response_1(prompt,ticket):
    model_output = llm(
      f"""
      Q: {prompt}
      Support ticket: {ticket}
      A:
      """,
      max_tokens=512, #Number of tokens the model should generate for this task.
      stop=["Q:", "\n"],
      temperature=0.001, #The value for temperature.
      echo=False,
    )

    temp_output = model_output["choices"][0]["text"]
    final_output = temp_output[temp_output.index('{'):]

    return final_output

In [19]:
prompt_1 = """
    You are an AI analyzing ticket text. Classify the tickets using one or more of the below mentioned categories only
    and not any other depending upon the content of the ticket complaint:
    - Hardware problem
    - Software problem
    - Connectivity failures
    - Data recovery

    Format the output as a JSON object with a single key-value pair as shown below:
    {"category": "your_category_prediction"}
"""

In [20]:
start = time.time()
data_1['model_response'] = data_1['support_ticket_text'].apply(lambda x: response_1(prompt_1, x))
end = time.time()


llama_print_timings:        load time =  102411.78 ms
llama_print_timings:      sample time =       6.04 ms /     9 runs   (    0.67 ms per token,  1490.56 tokens per second)
llama_print_timings: prompt eval time =  102410.52 ms /   162 tokens (  632.16 ms per token,     1.58 tokens per second)
llama_print_timings:        eval time =    7680.27 ms /     8 runs   (  960.03 ms per token,     1.04 tokens per second)
llama_print_timings:       total time =  110152.90 ms /   170 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =  102411.78 ms
llama_print_timings:      sample time =       6.75 ms /     9 runs   (    0.75 ms per token,  1334.12 tokens per second)
llama_print_timings: prompt eval time =   29938.43 ms /    51 tokens (  587.03 ms per token,     1.70 tokens per second)
llama_print_timings:        eval time =    8346.98 ms /     8 runs   ( 1043.37 ms per token,     0.96 tokens per second)
llama_print_timings:       total time =   38344.18 ms /    59 

In [21]:
print("Time taken ",(end-start))

Time taken  912.8417797088623


In [22]:
#Checking the first five rows of the data to confirm whether the new column was added
data_1['model_response'].head()

,model_response
0,"{""category"": ""Connectivity failures""}"
1,"{""category"": ""Hardware problem""}"
2,"{""category"": ""Data recovery""}"
3,"{""category"": ""Connectivity failures""}"
4,"{""category"": ""Hardware problem""}"


In [23]:
# A function to parse the JSON output from the model
def extract_json_data(json_str):
    try:
        # Find the indices of the opening and closing curly braces
        json_start = json_str.find('{')
        json_end = json_str.rfind('}')

        if json_start != -1 and json_end != -1:
            extracted_category = json_str[json_start:json_end + 1]  # Extract the JSON object
            data_dict = json.loads(extracted_category)
            return data_dict
        else:
            print(f"Warning: JSON object not found in response: {json_str}")
            return {}
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return {}

In [24]:
i = 12
print(data_1.loc[i, 'support_ticket_text'])

I accidentally spilled water on my laptop, and it won't turn on. Can you help me assess the damage and recover my data?


In [25]:
print(data_1.loc[i, 'model_response'])

{"category": "Data recovery"}


After running a few examples, the prompt works because it adds the right class.

In [26]:
data_1["model_response_parsed"] = data_1["model_response"].apply(extract_json_data)

In [27]:
data_1["model_response_parsed"]

,model_response_parsed
0,{'category': 'Connectivity failures'}
1,{'category': 'Hardware problem'}
2,{'category': 'Data recovery'}
3,{'category': 'Connectivity failures'}
4,{'category': 'Hardware problem'}
5,{'category': 'Connectivity failures'}
6,{'category': 'Software problem'}
7,{'category': 'Hardware problem'}
8,{'category': 'Data recovery'}
9,{'category': 'Hardware problem'}


In [28]:
data_1['model_response_parsed'].value_counts()

,count
model_response_parsed,
{'category': 'Data recovery'},7
{'category': 'Connectivity failures'},6
{'category': 'Hardware problem'},6
{'category': 'Software problem'},1
"{'category': 'Software problem, Data recovery'}",1


In [29]:
data_1[data_1["model_response_parsed"]=={}]

,support_tick_id,support_ticket_text,model_response,model_response_parsed


In [30]:
model_response_parsed_df_1 = pd.json_normalize(data_1["model_response_parsed"])
model_response_parsed_df_1.head()

,category
0,Connectivity failures
1,Hardware problem
2,Data recovery
3,Connectivity failures
4,Hardware problem


In [31]:
# Concatenate the two dataframes
data_with_parsed_model_output_1 = pd.concat([data_1, model_response_parsed_df_1], axis=1)
data_with_parsed_model_output_1.head()

,support_tick_id,support_ticket_text,model_response,model_response_parsed,category
0,ST2023-006,My internet connection has significantly slowe...,"{""category"": ""Connectivity failures""}",{'category': 'Connectivity failures'},Connectivity failures
1,ST2023-007,Urgent help required! My laptop refuses to sta...,"{""category"": ""Hardware problem""}",{'category': 'Hardware problem'},Hardware problem
2,ST2023-008,I've accidentally deleted essential work docum...,"{""category"": ""Data recovery""}",{'category': 'Data recovery'},Data recovery
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"{""category"": ""Connectivity failures""}",{'category': 'Connectivity failures'},Connectivity failures
4,ST2023-010,"My smartphone battery is draining rapidly, eve...","{""category"": ""Hardware problem""}",{'category': 'Hardware problem'},Hardware problem


In [32]:
final_data_1 = data_with_parsed_model_output_1.drop(['model_response','model_response_parsed'], axis=1)
final_data_1.head()

,support_tick_id,support_ticket_text,category
0,ST2023-006,My internet connection has significantly slowe...,Connectivity failures
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware problem
2,ST2023-008,I've accidentally deleted essential work docum...,Data recovery
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Connectivity failures
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware problem


##  Creating Tags

In [33]:
# creating a copy of the data
data_2 = ticket_data.copy()

In [34]:
def response_2(prompt,ticket,category):
    model_output = llm(
      f"""
      Q: {prompt}
      Support ticket: {ticket}
      Category: {category}
      A:
      """,
      max_tokens=512, #Setting the maximum number of tokens the model should generate for this task.
      stop=["Q:", "\n"],
      temperature=0.001, #Setting the value for temperature.
      echo=False,
    )

    temp_output = model_output["choices"][0]["text"]
    final_output = temp_output[temp_output.index('{'):]

    return final_output

In [35]:
prompt_2 = """
   You are an AI analyzing ticket text. Classify the tickets using one or more of the below mentioned categories only and not any other depending upon the content of the ticket complaint:
    - Hardware problem
    - Software problem
    - Connectivity failures
    - Data recovery

    Subsequently, add to each ticket one or more of the below mentioned keywords depending upon the content of the ticket text:
    - laptop
    - smartphone
    - USB
    - document loss
    - password
    - internet signal
    - battery
    - access problem

    Format the overall output as a JSON object with a key-value pair as shown below:
    {"category": "your_category_prediction", "tag": "your_tag_prediction"}

    Ensure that every ticket has at least one category and one tag.
"""

In [36]:
start = time.time()
data_2["model_response_2"]=final_data_1[['support_ticket_text','category']].apply(lambda x: response_2(prompt_2, x[0],x[1]),axis =1)
end = time.time()

Llama.generate: prefix-match hit

llama_print_timings:        load time =  102411.78 ms
llama_print_timings:      sample time =      11.03 ms /    17 runs   (    0.65 ms per token,  1541.39 tokens per second)
llama_print_timings: prompt eval time =  149191.77 ms /   249 tokens (  599.16 ms per token,     1.67 tokens per second)
llama_print_timings:        eval time =   15235.28 ms /    16 runs   (  952.20 ms per token,     1.05 tokens per second)
llama_print_timings:       total time =  164545.65 ms /   265 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =  102411.78 ms
llama_print_timings:      sample time =      10.93 ms /    17 runs   (    0.64 ms per token,  1554.78 tokens per second)
llama_print_timings: prompt eval time =   34159.68 ms /    58 tokens (  588.96 ms per token,     1.70 tokens per second)
llama_print_timings:        eval time =   13927.04 ms /    16 runs   (  870.44 ms per token,     1.15 tokens per second)
llama_print_timings:       to

In [37]:
print("Time taken ",end-start)

Time taken  1177.6256408691406


In [38]:
data_2['model_response_2'].head()

,model_response_2
0,"{""category"": ""Connectivity failures"", ""tag"": ""..."
1,"{""category"": ""Hardware problem"", ""tag"": ""laptop""}"
2,"{""category"": ""Data recovery"", ""tag"": ""document..."
3,"{""category"": ""Connectivity failures"", ""tag"": ""..."
4,"{""category"": ""Hardware problem"", ""tag"": ""batte..."


In [39]:
i = 13
print(data_2.loc[i, 'support_ticket_text'])

My USB flash drive is physically damaged, and I need assistance in recovering critical files from it.


In [40]:
print(data_2.loc[i, 'model_response_2'])

{"category": "Data recovery", "tag": "USB"}


In [41]:
# Applying the function to the model response
data_2['model_response_parsed_2'] = data_2['model_response_2'].apply(extract_json_data)

In [42]:
data_2["model_response_parsed_2"]

,model_response_parsed_2
0,"{'category': 'Connectivity failures', 'tag': '..."
1,"{'category': 'Hardware problem', 'tag': 'laptop'}"
2,"{'category': 'Data recovery', 'tag': 'document..."
3,"{'category': 'Connectivity failures', 'tag': '..."
4,"{'category': 'Hardware problem', 'tag': 'batte..."
5,"{'category': 'Connectivity failures', 'tag': '..."
6,"{'category': 'Software problem', 'tag': 'perfo..."
7,"{'category': 'Hardware problem', 'tag': 'laptop'}"
8,"{'category': 'Data recovery', 'tag': 'external..."
9,"{'category': 'Hardware problem', 'tag': 'lapto..."


In [43]:
# Normalizing the model_response_parsed column
model_response_parsed_df_2 = pd.json_normalize(data_2['model_response_parsed_2'])
model_response_parsed_df_2.head()

,category,tag
0,Connectivity failures,internet signal
1,Hardware problem,laptop
2,Data recovery,document loss
3,Connectivity failures,internet signal
4,Hardware problem,battery


In [44]:
# Concatinating two dataframes
data_with_parsed_model_output_2 = pd.concat([data_2, model_response_parsed_df_2], axis=1)
data_with_parsed_model_output_2.head()

,support_tick_id,support_ticket_text,model_response_2,model_response_parsed_2,category,tag
0,ST2023-006,My internet connection has significantly slowe...,"{""category"": ""Connectivity failures"", ""tag"": ""...","{'category': 'Connectivity failures', 'tag': '...",Connectivity failures,internet signal
1,ST2023-007,Urgent help required! My laptop refuses to sta...,"{""category"": ""Hardware problem"", ""tag"": ""laptop""}","{'category': 'Hardware problem', 'tag': 'laptop'}",Hardware problem,laptop
2,ST2023-008,I've accidentally deleted essential work docum...,"{""category"": ""Data recovery"", ""tag"": ""document...","{'category': 'Data recovery', 'tag': 'document...",Data recovery,document loss
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,"{""category"": ""Connectivity failures"", ""tag"": ""...","{'category': 'Connectivity failures', 'tag': '...",Connectivity failures,internet signal
4,ST2023-010,"My smartphone battery is draining rapidly, eve...","{""category"": ""Hardware problem"", ""tag"": ""batte...","{'category': 'Hardware problem', 'tag': 'batte...",Hardware problem,battery


In [45]:
# Dropping model_response and model_response_parsed columns
final_data_2 = data_with_parsed_model_output_2.drop(['model_response_2','model_response_parsed_2'], axis=1)
final_data_2.head()

,support_tick_id,support_ticket_text,category,tag
0,ST2023-006,My internet connection has significantly slowe...,Connectivity failures,internet signal
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware problem,laptop
2,ST2023-008,I've accidentally deleted essential work docum...,Data recovery,document loss
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Connectivity failures,internet signal
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware problem,battery


In [46]:
# Checking the value counts of Category column
final_data_2['tag'].value_counts()

,count
tag,
internet signal,5
laptop,5
document loss,2
"document loss, USB",2
performance issue,1
battery,1
external hard drive,1
"laptop, graphics card",1
USB,1


In [47]:
final_data_2 = final_data_2[["support_tick_id","support_ticket_text","category","tag"]]
final_data_2

,support_tick_id,support_ticket_text,category,tag
0,ST2023-006,My internet connection has significantly slowe...,Connectivity failures,internet signal
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware problem,laptop
2,ST2023-008,I've accidentally deleted essential work docum...,Data recovery,document loss
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Connectivity failures,internet signal
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware problem,battery
5,ST2023-011,I'm locked out of my online banking account an...,Connectivity failures,internet signal
6,ST2023-012,"My computer's performance is sluggish, severel...",Software problem,performance issue
7,ST2023-013,I'm experiencing a recurring blue screen error...,Hardware problem,laptop
8,ST2023-014,My external hard drive isn't being recognized ...,Data recovery,external hard drive
9,ST2023-015,The graphics card in my gaming laptop seems to...,Hardware problem,"laptop, graphics card"


## **Assigning Priority and ETA**

In [48]:
# creating a copy of the data
data_3 = final_data_2.copy()

In [49]:
def response_3(prompt,ticket,category,tag):
    model_output = llm(
      f"""
      Q: {prompt}
      Support ticket: {ticket}
      category: {category}
      tag: {tag}
      A:
      """,
      max_tokens=512,
      stop=["Q:"],
      temperature=0.0,
      echo=False,
    )

    temp_output = model_output["choices"][0]["text"]
    final_output = temp_output[temp_output.index('{'):]

    return final_output

In [50]:
prompt_3 = """
You are an IT service-desk triage assistant.

Using the existing ticket category and tags, assign an operational priority
and response ETA. Do not classify category or tags again.

Priority definitions:

HIGH URGENCY:
Assign High urgency only when the ticket contains concrete evidence of at
least one of the following:

1. An active security or account-compromise risk.
2. Ongoing data loss or physical damage that may become worse if the
   device continues to be used.
3. Complete loss of access to a business-critical system or device,
   with no reasonable workaround.
4. A specific near-term business deadline that will be missed unless
   the issue is addressed quickly.
5. Multiple users or an essential business service are affected.

NORMAL:
Assign Normal when the issue is inconvenient but there is no evidence of
immediate business, security, or data-loss impact. Examples include:

- Slow or intermittent connectivity when some service remains available.
- Battery drain, performance degradation, weak Wi-Fi, or peripheral problems.
- Problems with a reasonable temporary workaround.
- Requests where the user provides no evidence of an imminent deadline,
  active damage, security risk, or complete inability to work.

Important rules:

- Emotional words such as "urgent", "critical", "important", or "please help"
  are not sufficient by themselves for High urgency.
- A Data recovery category is not automatically High urgency.
- If the evidence is insufficient or ambiguous, choose Normal.
- Do not force a particular number or percentage of tickets into either class.
- Base the decision on operational impact, not the customer's tone.

ETA mapping:
- High urgency -> 1 business day
- Normal -> 2 to 3 business days

Return exactly one JSON object:

{
  "priority": "High urgency or Normal",
  "eta": "1 business day or 2 to 3 business days",
  "priority_reason": "Brief evidence-based explanation"
}
"""

**Note**: The output of the model should be in a structured format (JSON format).

In [51]:
# Applying generate_llama_response function on support_ticket_text column
start = time.time()
data_3['model_response_new'] = final_data_2[['support_ticket_text','category','tag']].apply(lambda x: response_3(prompt_3, x[0],x[1],x[2]),axis=1)
end = time.time()

Llama.generate: prefix-match hit

llama_print_timings:        load time =  102411.78 ms
llama_print_timings:      sample time =      58.32 ms /    89 runs   (    0.66 ms per token,  1525.98 tokens per second)
llama_print_timings: prompt eval time =  338784.18 ms /   556 tokens (  609.32 ms per token,     1.64 tokens per second)
llama_print_timings:        eval time =   79229.38 ms /    88 runs   (  900.33 ms per token,     1.11 tokens per second)
llama_print_timings:       total time =  418647.71 ms /   644 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =  102411.78 ms
llama_print_timings:      sample time =      44.52 ms /    60 runs   (    0.74 ms per token,  1347.83 tokens per second)
llama_print_timings: prompt eval time =   38226.92 ms /    63 tokens (  606.78 ms per token,     1.65 tokens per second)
llama_print_timings:        eval time =   63031.92 ms /    59 runs   ( 1068.34 ms per token,     0.94 tokens per second)
llama_print_timings:       to

In [52]:
print("Time taken ",(end-start))

Time taken  2620.853367090225


In [53]:
i = 10
print(data_3.loc[i, 'support_ticket_text'])

I accidentally formatted my USB drive with critical work files. Can you guide me through the data recovery process to retrieve these files?


In [54]:
print(data_3.loc[i, 'model_response_new'])

{
        "priority": "Normal",
        "eta": "2 to 3 business days",
        "priority_reason": "The data loss occurred due to user error, which does not meet the criteria for High urgency. The data can be recovered using standard data recovery tools and processes."
      }


In [55]:
# Applying the function to the model response
data_3['model_response_parsed_new'] = data_3['model_response_new'].apply(extract_json_data)
data_3['model_response_parsed_new'].head()

,model_response_parsed_new
0,"{'priority': 'Normal', 'eta': '2 to 3 business..."
1,"{'priority': 'High urgency', 'eta': '1 busines..."
2,"{'priority': 'High urgency', 'eta': '1 busines..."
3,"{'priority': 'Normal', 'eta': '2 to 3 business..."
4,"{'priority': 'Normal', 'eta': '2 to 3 business..."


In [56]:
# Normalizing the model_response_parsed column
model_response_parsed_df_3 = pd.json_normalize(data_3['model_response_parsed_new'])
model_response_parsed_df_3.head(10)

,priority,eta,priority_reason
0,Normal,2 to 3 business days,The user reports a significant slowdown in int...
1,High urgency,1 business day,The user has provided concrete evidence of a s...
2,High urgency,1 business day,The user has reported substantial data loss wh...
3,Normal,2 to 3 business days,The issue is inconvenient but there is no evid...
4,Normal,2 to 3 business days,The user reports a battery issue but there is ...
5,High urgency,1 business day,The user's inability to access their online ba...
6,Normal,2 to 3 business days,The user's productivity is impacted by a perfo...
7,Normal,2 to 3 business days,The user is experiencing a recurring blue scre...
8,Normal,2 to 3 business days,The data recovery category alone does not indi...
9,Normal,2 to 3 business days,The issue is a hardware problem affecting a si...


In [82]:
# Concatinating the two dataframes
data_with_parsed_model_output_3 = pd.concat([data_3, model_response_parsed_df_3], axis=1)
data_with_parsed_model_output_3.head()

,support_tick_id,support_ticket_text,category,tag,model_response_new,model_response_parsed_new,priority,eta,priority_reason
0,ST2023-006,My internet connection has significantly slowe...,Connectivity failures,internet signal,"{\n ""priority"": ""Normal"",\n ""e...","{'priority': 'Normal', 'eta': '2 to 3 business...",Normal,2 to 3 business days,The user reports a significant slowdown in int...
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware problem,laptop,"{\n ""priority"": ""High urgency"",\n ...","{'priority': 'High urgency', 'eta': '1 busines...",High urgency,1 business day,The user has provided concrete evidence of a s...
2,ST2023-008,I've accidentally deleted essential work docum...,Data recovery,document loss,"{\n ""priority"": ""High urgency"",\n ...","{'priority': 'High urgency', 'eta': '1 busines...",High urgency,1 business day,The user has reported substantial data loss wh...
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Connectivity failures,internet signal,"{\n ""priority"": ""Normal"",\n ""e...","{'priority': 'Normal', 'eta': '2 to 3 business...",Normal,2 to 3 business days,The issue is inconvenient but there is no evid...
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware problem,battery,"{\n ""priority"": ""Normal"",\n ""e...","{'priority': 'Normal', 'eta': '2 to 3 business...",Normal,2 to 3 business days,The user reports a battery issue but there is ...


In [83]:
final_data_3 = data_with_parsed_model_output_3[["support_tick_id","support_ticket_text","category", "tag", "priority", "eta"]]

In [84]:
final_data_3

,support_tick_id,support_ticket_text,category,tag,priority,eta
0,ST2023-006,My internet connection has significantly slowe...,Connectivity failures,internet signal,Normal,2 to 3 business days
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware problem,laptop,High urgency,1 business day
2,ST2023-008,I've accidentally deleted essential work docum...,Data recovery,document loss,High urgency,1 business day
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Connectivity failures,internet signal,Normal,2 to 3 business days
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware problem,battery,Normal,2 to 3 business days
5,ST2023-011,I'm locked out of my online banking account an...,Connectivity failures,internet signal,High urgency,1 business day
6,ST2023-012,"My computer's performance is sluggish, severel...",Software problem,performance issue,Normal,2 to 3 business days
7,ST2023-013,I'm experiencing a recurring blue screen error...,Hardware problem,laptop,Normal,2 to 3 business days
8,ST2023-014,My external hard drive isn't being recognized ...,Data recovery,external hard drive,Normal,2 to 3 business days
9,ST2023-015,The graphics card in my gaming laptop seems to...,Hardware problem,"laptop, graphics card",Normal,2 to 3 business days


## **Creating a Draft Response**

In [85]:
# creating a copy of the data
data_4 = final_data_3.copy()

In [86]:
def response_4(prompt,ticket,category,tags,priority,eta):
    model_output = llm(
      f"""
      Q: {prompt}
      Support ticket: {ticket}
      category : {category}
      tag : {tags}
      priority: {priority}
      eta: {eta}
      A:
      """,
      max_tokens=512, #Number of tokens
      stop=["Q:"],
      temperature=0.001, #Temperature.
      echo=False,
    )

    temp_output = model_output["choices"][0]["text"]


    return temp_output

In [87]:
prompt_4 = """
    You are an IT service-desk response assistant.

    The support ticket has already been analyzed in Task 3. The supplied category,
    tag, priority, and ETA are authoritative inputs. Do not classify, change, or
    contradict them. Your only task is to draft the customer response.

    Response requirements:
    1. Write a concise, professional response of three to five sentences.
    2. Begin by acknowledging the problem and apologizing for the inconvenience.
    3. Refer to the assigned ETA exactly as supplied:
       - High urgency: within 1 business day.
       - Normal: within 2 to 3 business days.
    4. Do not promise an immediate resolution or a faster response than the
       supplied ETA.
    5. For Connectivity failures, say that the support team will investigate the
       connection issue and request relevant device, router, or error details.
    6. For Hardware problems, explain that a technician may request diagnostic
       details to determine the appropriate solution.
    7. For Software problems, request the application, operating-system, error-message,
       and recent-change details that are relevant to troubleshooting.
    8. For Data recovery, advise the customer to stop using the affected storage
       device when continued use could overwrite or worsen lost data, and request
       relevant recovery details.
    9. Never request passwords, authentication codes, full payment details, or other
       sensitive credentials.
    10. Do not invent actions already taken, diagnoses, guarantees, or completion times.

    Return only the plain-text customer response. Do not return JSON, a label, an
    explanation of the classification, or surrounding quotation marks.
"""

**Note** : For this task, we will not be using the *`extract_json_data`* function. Hence, the output from the model should be a plain string and not a JSON object.

In [88]:
#Applying generate_llama_response function on support_ticket_text column using explicit column names
start = time.time()
data_4['model_response_4'] = data_4.apply(
    lambda row: response_4(
        prompt_4,
        row['support_ticket_text'],
        row['category'],
        row['tag'],
        row['priority'],
        row['eta']
    ),
    axis=1
)
end = time.time()

Llama.generate: prefix-match hit

llama_print_timings:        load time =  102411.78 ms
llama_print_timings:      sample time =      57.88 ms /    82 runs   (    0.71 ms per token,  1416.77 tokens per second)
llama_print_timings: prompt eval time =  297396.41 ms /   477 tokens (  623.47 ms per token,     1.60 tokens per second)
llama_print_timings:        eval time =   72427.20 ms /    81 runs   (  894.16 ms per token,     1.12 tokens per second)
llama_print_timings:       total time =  370353.14 ms /   558 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =  102411.78 ms
llama_print_timings:      sample time =      83.28 ms /   125 runs   (    0.67 ms per token,  1500.94 tokens per second)
llama_print_timings: prompt eval time =   48415.00 ms /    79 tokens (  612.85 ms per token,     1.63 tokens per second)
llama_print_timings:        eval time =  124488.48 ms /   124 runs   ( 1003.94 ms per token,     1.00 tokens per second)
llama_print_timings:       to

In [89]:
print("Time taken",(end-start))

Time taken 4188.941020965576


In [90]:
# Write the code to check the first five rows of the data to confirm whether the new column has been added
data_4['model_response_4'].head()

,model_response_4
0,"Dear Valued Customer,\n We apologize fo..."
1,"Dear Valued Customer,\n We sincerely ap..."
2,"Dear Valued Customer,\n \n We sincer..."
3,"Dear Valued Customer,\n We apologize fo..."
4,"Dear Valued Customer,\n We apologize fo..."


In [91]:
i = 4
print(data_4.loc[i, 'support_ticket_text'])

My smartphone battery is draining rapidly, even with minimal use. Can you help me identify and rectify this battery issue?


In [92]:
print(data_4.loc[i, 'model_response_4'])

 Dear Valued Customer,
       We apologize for the inconvenience you're experiencing with your smartphone battery draining rapidly despite minimal use. Our team will investigate this issue further and may request diagnostic details to determine the appropriate solution within the assigned ETA of 2 to 3 business days. Please do not attempt to replace the battery yourself as it may cause further damage to your device. Thank you for bringing this matter to our attention.


In [93]:
final_data_4 = pd.concat([final_data_3,data_4["model_response_4"]],axis=1)

In [94]:
final_data_4.rename(columns={"model_response_4":"response"},inplace=True)

In [95]:
final_data_4

,support_tick_id,support_ticket_text,category,tag,priority,eta,response
0,ST2023-006,My internet connection has significantly slowe...,Connectivity failures,internet signal,Normal,2 to 3 business days,"Dear Valued Customer,\n We apologize fo..."
1,ST2023-007,Urgent help required! My laptop refuses to sta...,Hardware problem,laptop,High urgency,1 business day,"Dear Valued Customer,\n We sincerely ap..."
2,ST2023-008,I've accidentally deleted essential work docum...,Data recovery,document loss,High urgency,1 business day,"Dear Valued Customer,\n \n We sincer..."
3,ST2023-009,Despite being in close proximity to my Wi-Fi r...,Connectivity failures,internet signal,Normal,2 to 3 business days,"Dear Valued Customer,\n We apologize fo..."
4,ST2023-010,"My smartphone battery is draining rapidly, eve...",Hardware problem,battery,Normal,2 to 3 business days,"Dear Valued Customer,\n We apologize fo..."
5,ST2023-011,I'm locked out of my online banking account an...,Connectivity failures,internet signal,High urgency,1 business day,"Dear Valued Customer,\n We apologize fo..."
6,ST2023-012,"My computer's performance is sluggish, severel...",Software problem,performance issue,Normal,2 to 3 business days,"Dear Valued Customer,\n \n We apolog..."
7,ST2023-013,I'm experiencing a recurring blue screen error...,Hardware problem,laptop,Normal,2 to 3 business days,"Dear Valued Customer,\n \n We apolog..."
8,ST2023-014,My external hard drive isn't being recognized ...,Data recovery,external hard drive,Normal,2 to 3 business days,"Dear Valued Customer,\n \n We apolog..."
9,ST2023-015,The graphics card in my gaming laptop seems to...,Hardware problem,"laptop, graphics card",Normal,2 to 3 business days,"Dear Valued Customer,\n We apologize fo..."


## **Model Output Analysis**

In [96]:
# Creating a copy of the dataframe of task-4
final_data = final_data_4.copy()

In [97]:
final_data['response'].value_counts()    # column name for the column containing ticket categories

,count
response,
"Dear Valued Customer,\n We apologize for the inconvenience you've experienced with your internet connection. Our team will investigate the connection issue as soon as possible within the given ETA of 2 to 3 business days. In the meantime, please provide any relevant device, router, or error details that may help expedite the resolution process. Thank you for your patience and understanding.",1
"Dear Valued Customer,\n We sincerely apologize for the inconvenience you're experiencing with your laptop not starting. Our team understands the urgency of your situation and will prioritize your request accordingly. As per the provided ETA, we aim to resolve this issue within 1 business day. A technician may reach out to you for additional diagnostic details to determine the appropriate solution. In the meantime, please ensure that you do not attempt to use the laptop to prevent any potential data loss. Thank you for bringing this matter to our attention, and we will keep you updated on the progress.",1
"Dear Valued Customer,\n \n We sincerely apologize for the data loss incident you have experienced. We understand the importance of your work documents and the urgency of this situation. Our team will prioritize the data recovery process as per the assigned ETA of 1 business day.\n \n To ensure the best possible outcome, we kindly request that you avoid using the affected storage device to prevent any further data loss. Once the recovery process is initiated, we will keep you updated on the progress.\n \n Thank you for bringing this matter to our attention, and we appreciate your patience as we work to resolve this issue for you.",1
"Dear Valued Customer,\n We apologize for the inconvenience you're experiencing with your Wi-Fi signal. Our team will investigate the connection issue as soon as possible within the given ETA of 2 to 3 business days. In the meantime, please provide any relevant device, router, or error details that may help us diagnose the problem more effectively. Thank you for bringing this to our attention.",1
"Dear Valued Customer,\n We apologize for the inconvenience you're experiencing with your smartphone battery draining rapidly despite minimal use. Our team will investigate this issue further and may request diagnostic details to determine the appropriate solution within the assigned ETA of 2 to 3 business days. Please do not attempt to replace the battery yourself as it may cause further damage to your device. Thank you for bringing this matter to our attention.",1
"Dear Valued Customer,\n We apologize for the inconvenience you're experiencing with accessing your online banking account. Our team is committed to resolving this issue as soon as possible, with a target resolution time of within 1 business day as per the assigned ETA. While we investigate the connection issue, please provide any relevant device, router, or error details that may help expedite the process. Thank you for your patience and understanding.",1
"Dear Valued Customer,\n \n We apologize for the inconvenience you're experiencing with your computer's performance. Our team will investigate this issue as soon as possible, within the assigned ETA of 2 to 3 business days. We kindly ask for your patience as we work to optimize your system for improved productivity.\n \n In the meantime, please provide the following details to help us troubleshoot the issue effectively:\n 1. Which specific application(s) are you experiencing the performance issue with?\n 2. What is the operating system version on your computer?\n 3. Can you please describe the error messages, if any, that you've encountered?\n 4. Have there been any recent changes to your system, such as software installations or updates?\n \n Thank you for bringing this matter to our attention, and we appreciate your cooperation as we work to resolve the issue.",1
"Dear Valued Customer,\n \n We apologize for the inconvenience you're experiencing with your laptop's recurring blue scre

In [98]:
final_data["priority"].value_counts() #the column name for the column containing the priorities of the ticket.

,count
priority,
Normal,15
High urgency,6


In [99]:
final_data["eta"].value_counts()  #the column name for the column containing ticket resolution ETA.

,count
eta,
2 to 3 business days,15
1 business day,6


In [100]:
final_data.groupby(['category', 'eta']).support_tick_id.count() # Group by based on the categories and ETA.

category               eta                 
Connectivity failures  1 business day          1
                       2 to 3 business days    4
Data recovery          1 business day          3
                       2 to 3 business days    4
Hardware problem       1 business day          1
                       2 to 3 business days    5
Software problem       2 to 3 business days    2
Name: support_tick_id, dtype: int64